In [1]:
import pandas as pd

data={"Formula_pretty":[],
      "A_ions":[],
      "B_ions":[],
      "C_ions":[],
      "A_charge":[],
      "B_charge":[],
      "C_charge":[],
      "A_ionic_radii":[],
      "B_ionic_radii":[],
      "C_ionic_radii":[]}

In [3]:
import requests
from bs4 import BeautifulSoup
from collections import defaultdict
# Target URL
URL = "http://abulafia.mt.ic.ac.uk/shannon/radius.php"

# Send request
response = requests.get(URL)
response.raise_for_status()

# Parse HTML
soup = BeautifulSoup(response.text, "lxml")

# Find the table (main ionic radii table)
table = soup.find("table")


# element → charge → coordination → radius
elements = dict()

rows = table.find_all("tr") # skip header

for row in rows:

    cols = [c.get_text(strip=True) for c in row.find_all("td")]
    
    if cols==['Ion', 'Charge', 'Coordination', 'Spin State', 'Crystal Radius', 'Ionic Radius', 'Key*']:
        continue

    # Ion | Charge | Coordination | Spin | Crystal R | Ionic R | Key
    if len(cols) > 6:
        element = cols[0]
        charge = cols[1]
        coordination = cols[2]
        ionic_radius = float(cols[5])

        elements[element]={"Charge":{charge:{coordination:ionic_radius}}}

    elif len(cols)>5:
        charge = cols[0]
        coordination = cols[1]
        ionic_radius = float(cols[4])
        elements[element]["Charge"][charge]={coordination:ionic_radius}
    elif len(cols)==5:
        coordination = cols[0]
        ionic_radius = float(cols[3])
        elements[element]["Charge"][charge][coordination]=ionic_radius
    else:
        continue




TOTAL ATTENUATION WITH COHERENT SCATTERING VALUES

In [4]:
nist_mu = {
    'Li': 0.05503, 'Na': 0.06100, 'K': 0.06216, 'Rb': 0.05689, 
    'Cs': 0.05854, 'Ba': 0.05803, 'Sr': 0.05714, 'Ca': 0.06388,
    'Mg': 0.06296, 'Al': 0.06146, 'Sc': 0.05985, 'Ti': 0.05891,
    'V': 0.05794, 'Cr': 0.05930, 'Mn': 0.05852, 'Fe': 0.05995,
    'Ni': 0.06160, 'Cu': 0.05901, 'Zn': 0.05941, 'Ga': 0.05767,
    'Ge': 0.05727, 'Ag': 0.05921, 'Y': 0.05795, 'Zr': 0.05810,
    'Nb': 0.05866, 'Mo': 0.05837, 'Tc': 0.05876, 'Ru': 0.05846,
    'Rh': 0.05894, 'Pd': 0.05850, 'Co': 0.05906, 'In': 0.05849,
    'Sn': 0.05800, 'Ta': 0.06566, 'W': 0.06618, 'Re': 0.06688,
    'Hf': 0.06502, 'Bi': 0.07214, 'O': 0.06372, 'F': 0.06037,
    'Cl': 0.06128, 'Br': 0.05728, 'I': 0.05841
}

In [5]:
import xraydb

for i in elements:
    try:
        
        if i in nist_mu:
            elements[i]["mu"]=nist_mu[i]
        else:
            continue
        
        elements[i]["AtomicNumber"]=xraydb.atomic_number(i)
        elements[i]["AtomicMass"]=xraydb.atomic_mass(i)
    except:
        continue

In [8]:
A_SITE=set("Cs, Rb, K, Na, Li, Ba, Ca, Sr".split(", "))
B_SITE=set("Mg, Ca, Sr, Ba, Sc, Ti, V, Cr, Mn, Fe, Co, Ni, Cu, Zn, Y, Zr, Nb, Mo, Tc, Ru, Rh, Pd, Ag, Hf, Ta, W, Re, Al, Ga, In, Sn, Bi, Ge".split(", "))


C_SITE=set(["O","F","Br","I","Cl"])
C_RADII={"O":elements["O"]["Charge"]["-2"]["VI"],
        "Br":elements["Br"]["Charge"]["-1"]["VI"],
        "F":elements["F"]["Charge"]["-1"]["VI"],
        "I":elements["I"]["Charge"]["-1"]["VI"],
        "Cl":elements["I"]["Charge"]["-1"]["VI"]}


GENERATION OF ALL POSSIBLE PEROVSKITES

In [9]:
import re

# Helper function to properly multiply formulas (e.g., CH6N * 2 -> C2H12N2)
def multiply_formula(formula, multiplier):
    tokens = re.findall(r'([A-Z][a-z]*)(\d*\.?\d*)', formula)
    res = ""
    for el, count in tokens:
        c = float(count) if count else 1.0
        c_new = c * multiplier
        c_str = "" if c_new == 1 else str(int(c_new)) if c_new.is_integer() else str(c_new)
        res += f"{el}{c_str}"
    return res

# ---------------------------------------------------------
# 1. ABC3
# ---------------------------------------------------------
for i in A_SITE:
    for j in B_SITE:
        if i == j: continue  
        
        for qA_str, props_A in elements[i]["Charge"].items():
            if "XII" not in props_A: continue 
            rA = props_A["XII"]
            qA = int(qA_str)
            
            for qB_str, props_B in elements[j]["Charge"].items():
                if "VI" not in props_B: continue 
                rB = props_B["VI"]
                qB = int(qB_str)
                
                for k in C_SITE:
                    qC = -2 if k=="O" else -1
                    rC = C_RADII[k]
                    
                    if qA + qB + (3 * qC) == 0:
                        data["Formula_pretty"].append(f"{i}{j}{k}3")
                        data["A_ions"].append(i)
                        data["B_ions"].append(j)
                        data["C_ions"].append(k)
                        data["A_ionic_radii"].append(rA)
                        data["B_ionic_radii"].append(rB)
                        data["C_ionic_radii"].append(rC)
                        data["A_charge"].append(qA)
                        data["B_charge"].append(qB)
                        data["C_charge"].append(qC)

# ---------------------------------------------------------
# 2. A2BC6
# ---------------------------------------------------------
for i in A_SITE:
    for j in B_SITE:
        if i == j: continue  
        for qA_str, props_A in elements[i]["Charge"].items():
            if "XII" not in props_A: continue 
            rA = props_A["XII"]
            qA = int(qA_str)

            for qB_str, props_B in elements[j]["Charge"].items():
                if "VI" not in props_B: continue 
                rB = props_B["VI"]
                qB = int(qB_str)

                for k in C_SITE:
                    qC = -2 if k=="O" else -1
                    rC = C_RADII[k]
                    
                    if 2*qA + qB + (6 * qC) == 0:

                        data["Formula_pretty"].append(f"{i}2{j}{k}6")
                        data["A_ions"].append(i)
                        data["B_ions"].append(j)
                        data["C_ions"].append(k)
                        data["A_ionic_radii"].append(rA)
                        data["B_ionic_radii"].append(rB)
                        data["C_ionic_radii"].append(rC)
                        data["A_charge"].append(qA)
                        data["B_charge"].append(qB)
                        data["C_charge"].append(qC)

# ---------------------------------------------------------
# 3. A2BB'C6
# ---------------------------------------------------------
for i in A_SITE:
    for j in B_SITE:
        if i == j: continue  
        for jj in B_SITE:
            # Enforce j < jj to prevent identical duplicates (e.g., Fe/Co and Co/Fe)

            if j == jj or i == jj or jj<j: continue    

            for qA_str, props_A in elements[i]["Charge"].items():
                if "XII" not in props_A: continue 
                rA = props_A["XII"]
                qA = int(qA_str)

                for qB_str, props_B in elements[j]["Charge"].items():
                    if "VI" not in props_B: continue 
                    rB1 = props_B["VI"]
                    qB1 = int(qB_str)

                    for qBB_str, props_BB in elements[jj]["Charge"].items():
                        if "VI" not in props_BB: continue 
                        
                        # Use isolated variables to prevent loop accumulation
                        avg_rB = (rB1 + props_BB["VI"]) / 2
                        total_qB = qB1 + int(qBB_str)    
                        
                        for k in C_SITE:
                            qC = -2 if k=="O" else -1
                            rC = C_RADII[k]
                            
                            if 2*qA + total_qB + (6 * qC) == 0:

                                data["Formula_pretty"].append(f"{i}2{j}{jj}{k}6")
                                data["A_ions"].append(i)
                                data["B_ions"].append(f"{j}; {jj}")
                                data["C_ions"].append(k)
                                data["A_ionic_radii"].append(rA)
                                data["B_ionic_radii"].append(avg_rB)
                                data["C_ionic_radii"].append(rC)
                                data["A_charge"].append(qA)
                                data["B_charge"].append(total_qB)
                                data["C_charge"].append(qC)

# ---------------------------------------------------------
# 4. AA'BB'C6
# ---------------------------------------------------------
for i in A_SITE:
    for ii in A_SITE:
        if i == ii or ii<i: continue # Prevent duplicates
        
        for j in B_SITE:
            if i == j or ii == j: continue  
            for jj in B_SITE:
                if j == jj or i == jj or ii == jj or jj<j: continue    
                
                for qA_str, props_A in elements[i]["Charge"].items():
                    if "XII" not in props_A: continue 
                    rA1 = props_A["XII"]
                    qA1 = int(qA_str)

                    for qAA_str, props_AA in elements[ii]["Charge"].items():
                        if "XII" not in props_AA: continue 
                        avg_rA = (rA1 + props_AA["XII"]) / 2
                        total_qA = qA1 + int(qAA_str)
                        
                        for qB_str, props_B in elements[j]["Charge"].items():
                            if "VI" not in props_B: continue 
                            rB1 = props_B["VI"]
                            qB1 = int(qB_str)

                            for qBB_str, props_BB in elements[jj]["Charge"].items():
                                if "VI" not in props_BB: continue 
                                avg_rB = (rB1 + props_BB["VI"]) / 2
                                total_qB = qB1 + int(qBB_str)    
                                
                                for k in C_SITE:
                                    qC = -2 if k=="O" else -1
                                    rC = C_RADII[k]
                                    
                                    if total_qA + total_qB + (6 * qC) == 0:

                                        data["Formula_pretty"].append(f"{i}{ii}{j}{jj}{k}6")
                                        data["A_ions"].append(f"{i}; {ii}")
                                        data["B_ions"].append(f"{j}; {jj}")
                                        data["C_ions"].append(k)
                                        data["A_ionic_radii"].append(avg_rA)
                                        data["B_ionic_radii"].append(avg_rB)
                                        data["C_ionic_radii"].append(rC)
                                        data["A_charge"].append(total_qA)
                                        data["B_charge"].append(total_qB)
                                        data["C_charge"].append(qC)

# ---------------------------------------------------------
# 5. A2BB'C3C'3
# ---------------------------------------------------------
for i in A_SITE:
    for j in B_SITE:
        if i == j: continue  
        for jj in B_SITE:
            if j == jj or i == jj or jj<j: continue    

            for qA_str, props_A in elements[i]["Charge"].items():
                if "XII" not in props_A: continue 
                rA = props_A["XII"]
                qA = int(qA_str)

                for qB_str, props_B in elements[j]["Charge"].items():
                    if "VI" not in props_B: continue 
                    rB1 = props_B["VI"]
                    qB1 = int(qB_str)

                    for qBB_str, props_BB in elements[jj]["Charge"].items():
                        if "VI" not in props_BB: continue 
                        avg_rB = (rB1 + props_BB["VI"]) / 2
                        total_qB = qB1 + int(qBB_str)    
                        
                        for k in C_SITE:
                            for kk in C_SITE:   
                                if k> kk: continue # Prevent duplicates

                                qC1 = -2 if k=="O" else -1
                                qC2 = -2 if kk=="O" else -1
                                
                                total_qC = qC1 + qC2
                                avg_rC = (C_RADII[k] + C_RADII[kk]) / 2
                                
                                # Formula: 2*A + B + B' + 3*(C + C') = 0
                                if 2*qA + total_qB + 3*total_qC == 0:

                                    data["Formula_pretty"].append(f"{i}2{j}{jj}{k}3{kk}3")
                                    data["A_ions"].append(i)
                                    data["B_ions"].append(f"{j}; {jj}")
                                    data["C_ions"].append(f"{k}; {kk}")
                                    data["A_ionic_radii"].append(rA)
                                    data["B_ionic_radii"].append(avg_rB)
                                    data["C_ionic_radii"].append(avg_rC)
                                    data["A_charge"].append(qA)
                                    data["B_charge"].append(total_qB)
                                    data["C_charge"].append(total_qC / 2) # Append avg charge, adjust if needed
print(len(data["A_ions"]))

39670


In [10]:
for i in data:
    print(i,len(data[i]))

Formula_pretty 39670
A_ions 39670
B_ions 39670
C_ions 39670
A_charge 39670
B_charge 39670
C_charge 39670
A_ionic_radii 39670
B_ionic_radii 39670
C_ionic_radii 39670


In [11]:
ml=pd.DataFrame(data)

In [12]:
import math
import regex as re
def tolerance_factor(row):
    return (row["A_ionic_radii"] + row["C_ionic_radii"]) / (math.sqrt(2) * (row["B_ionic_radii"] + row["C_ionic_radii"]))

def octahedral_factor(row):
    return row["B_ionic_radii"] / row["C_ionic_radii"]

def is_structurally_valid(row):
    t = row["Tolerance_factor"]
    mu = row["Octahedral_factor"]

    formula_split=dict(re.findall(r'([A-Z][a-z]*)(\d*\.?\d*)', row["Formula_pretty"]))
    if len(formula_split.keys())==3:
        return (0.71 <= t <= 1.107) and (0.414 <= mu <= 0.592)
    else:                                                                               #for Double perovskites.
        return (0.71 <= t <= 1.107) and (0.41 < mu)

    

In [13]:
ml["Tolerance_factor"]=ml.apply(tolerance_factor,axis=1)
ml["Octahedral_factor"]=ml.apply(octahedral_factor,axis=1)
ml["Valid_Structure"]=ml.apply(is_structurally_valid,axis=1)
print( ml['Valid_Structure'].value_counts())

Valid_Structure
True     21881
False    17789
Name: count, dtype: int64


USING ONLY PEROVSKITES THAT PASS TOLERANCE AND OCTAHEDRAL FACTOR PARAMETERS

In [14]:
ml=ml[ml["Valid_Structure"]==True]


TEF CALCULATION AND EXPORTING DATA TO Generated_Data.csv

In [15]:
import math
import re


TISSUE_ZEFF = 7.42        # soft tissue reference
M_EXPONENT = 2.94         # photon interaction exponent

def compute_mass_fractions(composition):
    """
    composition: dict → {"Ag": 1, "Br": 3}
    element_data: ionic_data dictionary
    """
    total_mass = sum(count
        for el, count in composition.items()
    )

    mass_fractions = {
        el: (count) / total_mass
        for el, count in composition.items()
    }

    return mass_fractions


def compute_zeff(composition, m=M_EXPONENT):
    #Computes Z_eff using μ/ρ-weighted formula

    f = compute_mass_fractions(composition)

    numerator = 0.0
    denominator = 0.0

    for el, frac in f.items():
        Z = elements[el]["AtomicNumber"]
        A = elements[el]["AtomicMass"]
        mu = elements[el]["mu"]   # μ/ρ from XCOM

        numerator += frac * A * mu
        denominator += frac * (A / Z) * mu

    return numerator / denominator


def compute_tef(row):
    """
    Returns (Z_eff, TEF)
    """
    A=row["A_ions"].split("; ")
    B=row["B_ions"].split("; ")
    C=row["C_ions"].split("; ")
    
    composition=dict(re.findall(r'([A-Z][a-z]*)(\d*\.?\d*)', row["Formula_pretty"]))
    for i in composition:
        if composition[i]=="":
            composition[i]=1
        else:
            composition[i]=float(composition[i])

    try:
        zeff = compute_zeff(composition)
        tef = zeff
    except:
        print(row["Formula_pretty"],composition)
        for i in composition:
            print(i,elements[i])
    return tef
ml["TEF"]=ml.apply(compute_tef,axis=1)

ml.to_csv("OUTPUT DATA/Generated_Data.csv",index=False)

In [16]:
print(max(ml["TEF"]),min(ml["TEF"]))

58.097519404448796 10.000342404893379
